# 03_silver

Joins all three clean bronze tables into one unified Silver Delta table.
One row per (ticker, date) covering every trading day each stock was in
the index.

This is the single source of truth for all downstream feature engineering
and model training.

Input tables:
  precursor.bronze.alpaca_clean   (~917,749 rows)
  precursor.bronze.fred_clean     (~12,387 rows)
  precursor.bronze.sec_clean      (~225,346 rows)
  precursor.bronze.universe       (~1,417,256 rows)

Output table:
  precursor.silver.joined

In [0]:
%pip install "numpy<2.0" --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DoubleType, LongType,
    TimestampType, BooleanType, IntegerType,
)
from pyspark.sql.window import Window
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.silver")

logger.info("Silver pipeline initialised.")

INFO:precursor.silver:Silver pipeline initialised.


## Cell 3 — Load all input tables

In [0]:
def load_bronze_tables() -> tuple[DataFrame, DataFrame, DataFrame, DataFrame]:
    """Load all four bronze input tables and log row counts.

    Returns:
        Tuple of (alpaca_clean, fred_clean, sec_clean, universe).
    """
    alpaca   = spark.table("precursor.bronze.alpaca_clean")
    fred     = spark.table("precursor.bronze.fred_clean")
    sec      = spark.table("precursor.bronze.sec_clean")
    universe = spark.table("precursor.bronze.universe")

    logger.info("alpaca_clean   : %d rows", alpaca.count())
    logger.info("fred_clean     : %d rows", fred.count())
    logger.info("sec_clean      : %d rows", sec.count())
    logger.info("universe       : %d rows", universe.count())

    return alpaca, fred, sec, universe

## Cell 4 — Prepare FRED for joining

In [0]:
def prepare_fred(
    fred_df: DataFrame,
    universe_df: DataFrame,
) -> DataFrame:
    """Pivot FRED series from long format to one row per trading date.

    FRED arrives as one row per (series_id, date). We pivot so each series
    becomes a column, producing one row per date. We only keep trading days
    (sourced from universe) so the result aligns with alpaca_clean's date spine.

    Forward-filling after pivot is a safety net — bootstrap already forward-filled
    monthly series, but the pivot can introduce nulls on trading days that fall
    between FRED publication dates.

    Args:
        fred_df:     Spark DataFrame from precursor.bronze.fred_clean.
        universe_df: Spark DataFrame from precursor.bronze.universe.

    Returns:
        Pivoted DataFrame with one row per trading date and one column per
        FRED series.
    """
    # Restrict to trading days only — non-trading days have no alpaca rows
    # to join against, so including them would produce orphan FRED rows.
    trading_dates = (
        universe_df
        .filter(F.col("is_trading_day") == True)
        .select("date")
        .distinct()
    )

    fred_filtered = fred_df.join(trading_dates, on="date", how="inner")
    logger.info("fred after trading-day filter: %d rows", fred_filtered.count())

    # Pivot: one column per series_id.
    series_ids = ["DFF", "UNRATE", "CPIAUCSL", "T10Y2Y", "VIXCLS", "M2SL"]
    pivoted = (
        fred_filtered
        .groupBy("date")
        .pivot("series_id", series_ids)
        .agg(F.first("value"))
    )

    # Forward-fill each FRED column within the date-ordered spine.
    # Safety net for any trading days that fall between monthly publications.
    date_window = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
    for col in series_ids:
        pivoted = pivoted.withColumn(
            col,
            F.last(F.col(col), ignorenulls=True).over(date_window),
        )

    logger.info("fred pivoted: %d rows, %d columns.", pivoted.count(), len(pivoted.columns))
    return pivoted

## Cell 5 — Prepare SEC for joining

In [0]:
def prepare_sec(sec_df: DataFrame) -> DataFrame:
    """Aggregate SEC Form 4 filings to one row per (ticker, date).

    SEC arrives as one row per filing. We aggregate to filing counts per
    (ticker, transaction_date) so it can be left-joined onto alpaca_clean.
    Most (ticker, date) pairs will have zero filings — those are handled
    by the left join and null-fill in join_all_sources.

    We exclude invalid and very-late filings here to avoid polluting the
    filing_count signal with data-quality noise.

    Args:
        sec_df: Spark DataFrame from precursor.bronze.sec_clean.

    Returns:
        Aggregated DataFrame with columns [ticker, date, filing_count].
    """
    before = sec_df.count()
    logger.info("sec_clean input to prepare_sec: %d rows", before)

    # Exclude filings with invalid date order or excessive filing lag.
    # Very late filings may represent amended/restated filings that don't
    # correspond to real-time insider signals.
    filtered = sec_df.filter(
        (F.col("date_order_valid") == True) &
        (F.col("very_late_filing") == False)
    )

    agg = (
        filtered
        .groupBy("ticker", "transaction_date")
        .agg(F.count("accession_number").cast(IntegerType()).alias("filing_count"))
        .withColumnRenamed("transaction_date", "date")
    )

    after = agg.count()
    logger.info(
        "sec aggregated: %d distinct (ticker, date) combinations (from %d filings).",
        after, before,
    )
    return agg

## Cell 6 — Join all sources

In [0]:
def join_all_sources(
    alpaca_df: DataFrame,
    fred_pivoted_df: DataFrame,
    sec_agg_df: DataFrame,
    universe_df: DataFrame,
) -> DataFrame:
    """Join all bronze sources into the silver table at (ticker, date) grain.

    Join strategy:
      Base:        alpaca_clean — one row per (ticker, date) on trading days.
      Inner join:  universe — adds sector, confirms index membership + trading day.
      Left join:   fred_pivoted — macro data by date (same for all tickers).
      Left join:   sec_agg — insider filings by (ticker, date); most rows = 0.

    Inner joins shrink the result (stricter) — used when we only want rows that
    exist in both tables. Left joins preserve the base — used when the joining
    table is supplementary and missing data is expected and handled.

    Args:
        alpaca_df:       Spark DataFrame from precursor.bronze.alpaca_clean.
        fred_pivoted_df: Pivoted FRED DataFrame from prepare_fred().
        sec_agg_df:      Aggregated SEC DataFrame from prepare_sec().
        universe_df:     Spark DataFrame from precursor.bronze.universe.

    Returns:
        Joined DataFrame at (ticker, date) grain.
    """
    base_count = alpaca_df.count()
    logger.info("Join base (alpaca_clean): %d rows", base_count)

    # Step 1 — Filter universe to index members on trading days only.
    # We take sector from universe as it is not in alpaca_clean.
    universe_slim = (
        universe_df
        .filter(
            (F.col("is_trading_day") == True) &
            (F.col("in_index") == True)
        )
        .select("ticker", "date", "sector")
        .distinct()
    )

    # Step 2 — Inner join universe onto alpaca.
    # Inner join: we only want rows where the stock was in the index AND it
    # was a trading day. Any alpaca rows outside these criteria are noise.
    df = alpaca_df.join(universe_slim, on=["ticker", "date"], how="inner")
    after_universe = df.count()
    logger.info("After universe inner join: %d rows", after_universe)
    if after_universe > base_count:
        logger.warning(
            "Fan-out detected after universe join: %d → %d rows. Deduplicating.",
            base_count, after_universe,
        )
        dedup_w = Window.partitionBy("ticker", "date").orderBy(F.desc("processed_at"))
        df = (
            df
            .withColumn("_rn", F.row_number().over(dedup_w))
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )

    # Step 3 — Left join FRED pivoted on date.
    # Left join: every alpaca row must survive regardless of whether FRED has
    # a value for that date. FRED covers the full date range so nulls here
    # indicate a data gap, not a structural mismatch.
    df = df.join(fred_pivoted_df, on="date", how="left")
    after_fred = df.count()
    logger.info("After FRED left join: %d rows", after_fred)
    if after_fred != after_universe:
        logger.warning(
            "Row count changed after FRED join: %d → %d. Deduplicating.",
            after_universe, after_fred,
        )
        dedup_w = Window.partitionBy("ticker", "date").orderBy(F.desc("processed_at"))
        df = (
            df
            .withColumn("_rn", F.row_number().over(dedup_w))
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )

    # Step 4 — Left join SEC aggregated on (ticker, date).
    # Left join: most (ticker, date) pairs have no filings. Null filing_count
    # means zero filings — filled with 0 immediately after join.
    df = df.join(sec_agg_df, on=["ticker", "date"], how="left")
    df = df.withColumn("filing_count", F.coalesce(F.col("filing_count"), F.lit(0)))
    after_sec = df.count()
    logger.info("After SEC left join: %d rows", after_sec)
    if after_sec != after_fred:
        logger.warning(
            "Row count changed after SEC join: %d → %d. Deduplicating.",
            after_fred, after_sec,
        )
        dedup_w = Window.partitionBy("ticker", "date").orderBy(F.desc("processed_at"))
        df = (
            df
            .withColumn("_rn", F.row_number().over(dedup_w))
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )

    # Step 5 — Add joined_at metadata.
    df = df.withColumn("joined_at", F.current_timestamp())

    # Step 6 — Select final columns in clean order.
    fred_cols = ["DFF", "UNRATE", "CPIAUCSL", "T10Y2Y", "VIXCLS", "M2SL"]
    final_cols = (
        # Identity
        ["ticker", "date", "sector"] +
        # Price
        ["open", "high", "low", "close", "volume", "vwap",
         "adjusted", "ohlc_valid", "zero_volume", "extreme_move"] +
        # Macro
        fred_cols +
        # Insider
        ["filing_count"] +
        # Metadata
        ["joined_at"]
    )
    df = df.select(*final_cols)

    final_count = df.count()
    logger.info("Silver join complete: %d rows, %d columns.", final_count, len(df.columns))
    return df

## Cell 7 — Handle nulls after joining

In [0]:
def fill_nulls(joined_df: DataFrame) -> DataFrame:
    """Forward-fill FRED columns and verify no critical nulls remain.

    FRED columns should already be filled from bronze cleaning, but the pivot
    and join can introduce nulls on days where FRED has no published value.
    We forward-fill as a final safety net.

    Price columns (open, high, low, vwap) were filled with close in bronze
    cleaning — we verify here but do not re-fill.

    filing_count nulls were already replaced with 0 in the join step.

    Args:
        joined_df: Spark DataFrame from join_all_sources().

    Returns:
        DataFrame with nulls resolved.
    """
    fred_cols    = ["DFF", "UNRATE", "CPIAUCSL", "T10Y2Y", "VIXCLS", "M2SL"]
    price_cols   = ["open", "high", "low", "vwap"]
    critical_cols = ["close", "DFF", "VIXCLS"]

    df = joined_df

    # Forward-fill FRED columns ordered by date.
    # FRED is the same for all tickers so we partition by nothing — one global
    # time-ordered fill is correct.
    date_window = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
    for col in fred_cols:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            logger.info("fill_nulls: forward-filling %d nulls in %s.", null_count, col)
            df = df.withColumn(col, F.last(F.col(col), ignorenulls=True).over(date_window))

    # Verify price columns — these should have been handled in bronze.
    for col in price_cols:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            logger.warning("fill_nulls: %d unexpected nulls in %s — filling with close.", null_count, col)
            df = df.withColumn(col, F.coalesce(F.col(col), F.col("close")))

    # Verify filing_count.
    fc_nulls = df.filter(F.col("filing_count").isNull()).count()
    if fc_nulls > 0:
        logger.warning("fill_nulls: %d unexpected nulls in filing_count — filling with 0.", fc_nulls)
        df = df.withColumn("filing_count", F.coalesce(F.col("filing_count"), F.lit(0)))

    # Log warnings for any remaining nulls in critical columns.
    for col in critical_cols:
        remaining = df.filter(F.col(col).isNull()).count()
        if remaining > 0:
            logger.warning("fill_nulls: %d unfillable nulls remain in critical column %s.", remaining, col)
        else:
            logger.info("fill_nulls: %s — no nulls remaining.", col)

    return df

## Cell 8 — Write silver table

In [0]:
def write_silver(df: DataFrame) -> None:
    """Write the joined silver DataFrame to precursor.silver.joined.

    Args:
        df: Final joined and null-filled DataFrame.
    """
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.silver.joined")
    )

    result = spark.table("precursor.silver.joined")
    row_count     = result.count()
    col_count     = len(result.columns)
    date_range    = result.agg(F.min("date"), F.max("date")).collect()[0]
    ticker_count  = result.select("ticker").distinct().count()

    logger.info(
        "precursor.silver.joined written: %d rows, %d columns, %d tickers, %s → %s.",
        row_count, col_count, ticker_count,
        date_range[0], date_range[1],
    )

    display(spark.sql("""
        SELECT ticker, date, close, DFF, VIXCLS, filing_count, sector
        FROM precursor.silver.joined
        WHERE ticker = 'AAPL'
        ORDER BY date DESC
        LIMIT 10
    """))

## Cell 9 — Main execution

In [0]:
_pipeline_start = datetime.now()
logger.info("=== precursor.03_silver START at %s ===", _pipeline_start.isoformat())

try:
    logger.info("Step 1: Loading bronze tables.")
    _alpaca, _fred, _sec, _universe = load_bronze_tables()
    _alpaca_count  = _alpaca.count()
    _fred_count    = _fred.count()
    _sec_count     = _sec.count()

    logger.info("Step 2: Preparing FRED.")
    _fred_pivoted = prepare_fred(_fred, _universe)

    logger.info("Step 3: Preparing SEC.")
    _sec_agg = prepare_sec(_sec)

    logger.info("Step 4: Joining all sources.")
    _joined = join_all_sources(_alpaca, _fred_pivoted, _sec_agg, _universe)

    logger.info("Step 5: Filling nulls.")
    _filled = fill_nulls(_joined)

    logger.info("Step 6: Writing silver table.")
    write_silver(_filled)

    _silver      = spark.table("precursor.silver.joined")
    _silver_count = _silver.count()
    _col_count    = len(_silver.columns)
    _date_range   = _silver.agg(F.min("date").alias("mn"), F.max("date").alias("mx")).collect()[0]
    _ticker_count = _silver.select("ticker").distinct().count()

    _elapsed = (datetime.now() - _pipeline_start).total_seconds() / 60
    logger.info("=== precursor.03_silver END — %.2f min total ===", _elapsed)

    print("=" * 60)
    print("  PRECURSOR — SILVER PIPELINE SUMMARY")
    print("=" * 60)
    print("  Input rows:")
    print(f"    alpaca_clean   : {_alpaca_count:>10,}")
    print(f"    fred_clean     : {_fred_count:>10,}")
    print(f"    sec_clean      : {_sec_count:>10,}")
    print("  Output rows:")
    print(f"    silver.joined  : {_silver_count:>10,}")
    print(f"  Columns in output: {_col_count}")
    print(f"  Distinct tickers : {_ticker_count}")
    print(f"  Date range       : {_date_range['mn']} to {_date_range['mx']}")
    print(f"  Elapsed time     : {_elapsed:.2f} min")
    print("=" * 60)

except Exception as exc:
    logger.error("Silver pipeline failed: %s", exc, exc_info=True)

INFO:precursor.silver:=== precursor.03_silver START at 2026-05-01T19:07:39.613438 ===
INFO:precursor.silver:Step 1: Loading bronze tables.
INFO:precursor.silver:alpaca_clean   : 917749 rows
INFO:precursor.silver:fred_clean     : 12387 rows
INFO:precursor.silver:sec_clean      : 225346 rows
INFO:precursor.silver:universe       : 1417256 rows
INFO:precursor.silver:Step 2: Preparing FRED.
INFO:precursor.silver:fred after trading-day filter: 9482 rows
/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
INFO:precursor.silver:fred pivoted: 1590 rows, 7 columns.
INFO:precursor.silver:Step 3: Preparing SEC.
INFO:precursor.silver:sec_clean input to prepare_sec: 225346 rows
INFO:precursor.silver:sec aggregated: 77202 distinct (ticker, date) combinations (from 225346 filings).
INF

ticker,date,close,DFF,VIXCLS,filing_count,sector
AAPL,2026-04-30,271.35,3.64,16.89,0,Information Technology
AAPL,2026-04-29,270.17,3.64,18.81,0,Information Technology
AAPL,2026-04-28,270.71,3.64,17.83,0,Information Technology
AAPL,2026-04-27,267.61,3.64,18.02,0,Information Technology
AAPL,2026-04-24,271.06,3.64,18.71,0,Information Technology
AAPL,2026-04-23,273.43,3.64,19.31,1,Information Technology
AAPL,2026-04-22,273.17,3.64,18.92,0,Information Technology
AAPL,2026-04-21,266.17,3.64,19.5,0,Information Technology
AAPL,2026-04-20,273.05,3.64,18.87,0,Information Technology
AAPL,2026-04-17,270.23,3.64,17.48,0,Information Technology


INFO:precursor.silver:=== precursor.03_silver END — 0.58 min total ===


  PRECURSOR — SILVER PIPELINE SUMMARY
  Input rows:
    alpaca_clean   :    917,749
    fred_clean     :     12,387
    sec_clean      :    225,346
  Output rows:
    silver.joined  :    793,717
  Columns in output: 21
  Distinct tickers : 613
  Date range       : 2020-01-02 to 2026-04-30
  Elapsed time     : 0.58 min


## Cell 10 — Validation

In [0]:
logger.info("=== Running silver validation ===")

_checks: list[tuple[str, bool, str]] = []

def _check(name: str, passed: bool, detail: str = "") -> None:
    status = "PASS" if passed else "FAIL"
    msg    = f"[{status}] {name}" + (f" — {detail}" if detail else "")
    logger.info(msg)
    _checks.append((name, passed, detail))


try:
    _sv = spark.table("precursor.silver.joined")
    _sv_count = _sv.count()

    # 1. Silver row count is <= alpaca_clean (no fan-out) and > 700,000 (not too much lost).
    _alpaca_count_val = spark.table("precursor.bronze.alpaca_clean").count()
    _check(
      "silver <= alpaca_clean (no fan-out)",
      _sv_count <= _alpaca_count_val,
      f"silver={_sv_count:,}  alpaca={_alpaca_count_val:,}",
    )
    _check(
      "silver > 700,000 (no catastrophic loss)",
      _sv_count > 700_000,
      f"silver={_sv_count:,}",
    )

    # 2. No null close prices.
    _null_close = _sv.filter(F.col("close").isNull()).count()
    _check("No null close prices", _null_close == 0, f"{_null_close} nulls")

    # 3. No null DFF values.
    _null_dff = _sv.filter(F.col("DFF").isNull()).count()
    _check("No null DFF values", _null_dff == 0, f"{_null_dff} nulls")

    # 4. No null VIXCLS values.
    _null_vix = _sv.filter(F.col("VIXCLS").isNull()).count()
    _check("No null VIXCLS values", _null_vix == 0, f"{_null_vix} nulls")

    # 5. filing_count >= 0 for all rows.
    _neg_filings = _sv.filter(F.col("filing_count") < 0).count()
    _check("filing_count >= 0 for all rows", _neg_filings == 0, f"{_neg_filings} negative rows")

    # 6. Distinct tickers same as alpaca_clean.
    _sv_tickers    = _sv.select("ticker").distinct().count()
    _alpaca_tickers = spark.table("precursor.bronze.alpaca_clean").select("ticker").distinct().count()
    _check(
        "Distinct tickers matches alpaca_clean",
        _sv_tickers == _alpaca_tickers,
        f"silver={_sv_tickers}  alpaca={_alpaca_tickers}",
    )

    # 7. All dates are trading days per universe.
    _non_trading = (
        _sv
        .join(
            spark.table("precursor.bronze.universe")
            .filter(F.col("is_trading_day") == True)
            .select("ticker", "date"),
            on=["ticker", "date"],
            how="left_anti",
        )
        .count()
    )
    _check("All dates are trading days", _non_trading == 0, f"{_non_trading} non-trading rows")

    # 8. Sector is not null for any row.
    _null_sector = _sv.filter(F.col("sector").isNull()).count()
    _check("No null sector values", _null_sector == 0, f"{_null_sector} nulls")

    # 9. All 11 GICS sectors represented and no sector > 40% of rows.
    _sector_counts = (
        _sv
        .groupBy("sector")
        .count()
        .withColumn("pct", F.col("count") / _sv_count * 100)
        .orderBy(F.desc("count"))
        .collect()
    )
    _sector_n       = len(_sector_counts)
    _max_sector_pct = max(r["pct"] for r in _sector_counts) if _sector_counts else 0
    _check(
        "All 11 GICS sectors represented",
        _sector_n >= 11,
        f"{_sector_n} sectors found",
    )
    _check(
        "No sector > 40% of total rows",
        _max_sector_pct <= 40,
        f"max sector share = {_max_sector_pct:.1f}%",
    )

    # 10. Date range spans 2020-01-01 to today.
    _date_row = _sv.agg(F.min("date").alias("mn"), F.max("date").alias("mx")).collect()[0]
    _check(
        "Date range spans 2020 to current year",
        _date_row["mn"] is not None
        and str(_date_row["mn"]) <= "2020-01-31"
        and _date_row["mx"].year >= datetime.today().year,
        f"{_date_row['mn']} → {_date_row['mx']}",
    )

except Exception as exc:
    logger.error("Validation query failed: %s", exc, exc_info=True)
    _checks.append(("Validation queries executed without error", False, str(exc)))

_passed_n = sum(1 for _, p, _ in _checks if p)
_failed_n = len(_checks) - _passed_n

print("=" * 60)
print("  SILVER VALIDATION RESULTS")
print("=" * 60)
for name, passed, detail in _checks:
    status = "PASS" if passed else "FAIL"
    line   = f"  [{status}] {name}"
    if detail:
        line += f"\n         {detail}"
    print(line)
print("-" * 60)
print(f"  {_passed_n} passed  /  {_failed_n} failed")
if _failed_n > 0:
    print("  WARNING: validation failures detected — review logs above.")
print("=" * 60)

INFO:precursor.silver:=== Running silver validation ===
INFO:precursor.silver:[PASS] silver <= alpaca_clean (no fan-out) — silver=793,717  alpaca=917,749
INFO:precursor.silver:[PASS] silver > 700,000 (no catastrophic loss) — silver=793,717
INFO:precursor.silver:[PASS] No null close prices — 0 nulls
INFO:precursor.silver:[PASS] No null DFF values — 0 nulls
INFO:precursor.silver:[PASS] No null VIXCLS values — 0 nulls
INFO:precursor.silver:[PASS] filing_count >= 0 for all rows — 0 negative rows
INFO:precursor.silver:[PASS] Distinct tickers matches alpaca_clean — silver=613  alpaca=613
INFO:precursor.silver:[PASS] All dates are trading days — 0 non-trading rows
INFO:precursor.silver:[PASS] No null sector values — 0 nulls
INFO:precursor.silver:[PASS] All 11 GICS sectors represented — 11 sectors found
INFO:precursor.silver:[PASS] No sector > 40% of total rows — max sector share = 15.0%
INFO:precursor.silver:[PASS] Date range spans 2020 to current year — 2020-01-02 → 2026-04-30


  SILVER VALIDATION RESULTS
  [PASS] silver <= alpaca_clean (no fan-out)
         silver=793,717  alpaca=917,749
  [PASS] silver > 700,000 (no catastrophic loss)
         silver=793,717
  [PASS] No null close prices
         0 nulls
  [PASS] No null DFF values
         0 nulls
  [PASS] No null VIXCLS values
         0 nulls
  [PASS] filing_count >= 0 for all rows
         0 negative rows
  [PASS] Distinct tickers matches alpaca_clean
         silver=613  alpaca=613
  [PASS] All dates are trading days
         0 non-trading rows
  [PASS] No null sector values
         0 nulls
  [PASS] All 11 GICS sectors represented
         11 sectors found
  [PASS] No sector > 40% of total rows
         max sector share = 15.0%
  [PASS] Date range spans 2020 to current year
         2020-01-02 → 2026-04-30
------------------------------------------------------------
  12 passed  /  0 failed


In [0]:
spark.sql("""
    SELECT sector, COUNT(*) as cnt
    FROM precursor.silver.joined
    GROUP BY sector
    ORDER BY cnt DESC
""").display()

sector,cnt
Industrials,112698
Financials,107325
Information Technology,92331
Health Care,87809
,81308
Consumer Discretionary,68544
Consumer Staples,53188
Utilities,46859
Real Estate,46627
Materials,38990


In [0]:
spark.sql("""
    SELECT ticker, sector, COUNT(*) as cnt
    FROM precursor.silver.joined
    WHERE sector IS NULL OR sector = ''
    GROUP BY ticker, sector
    ORDER BY cnt DESC
    LIMIT 20
""").display()

ticker,sector,cnt
HOLX,,1568
LW,,1558
PAYC,,1541
MHK,,1498
LKQ,,1498
K,,1491
IPG,,1482
EMN,,1465
KMX,,1463
MKTX,,1434


In [0]:
from pyspark.sql import functions as F

SECTOR_PATCHES = {
    "HOLX": "Health Care",
    "LW":   "Consumer Staples",
    "PAYC": "Information Technology",
    "MHK":  "Consumer Discretionary",
    "LKQ":  "Consumer Discretionary",
    "K":    "Consumer Staples",
    "IPG":  "Communication Services",
    "EMN":  "Materials",
    "KMX":  "Consumer Discretionary",
    "MKTX": "Financials",
    "WBA":  "Consumer Staples",
    "HES":  "Energy",
    "ANSS": "Information Technology",
    "JNPR": "Information Technology",
    "DFS":  "Financials",
    "BWA":  "Consumer Discretionary",
    "FMC":  "Materials",
    "TFX":  "Health Care",
    "CE":   "Materials",
    "QRVO": "Information Technology"
}

# Read the existing silver table
df = spark.read.table("precursor.silver.joined")

# Apply patches
patch_expr = F.col("sector")
for ticker, sector in SECTOR_PATCHES.items():
    patch_expr = F.when(
        (F.col("ticker") == ticker) &
        (F.col("sector").isNull() | (F.col("sector") == "")),
        sector
    ).otherwise(patch_expr)

df = df.withColumn("sector", patch_expr)

# Verify
blank_count = df.filter(
    F.col("sector").isNull() | (F.col("sector") == "")
).count()
print(f"Blank sectors remaining: {blank_count}")

# Overwrite if clean
if blank_count == 0:
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("mergeSchema", "true") \
      .saveAsTable("precursor.silver.joined")
    print("✅ Silver table patched successfully")
else:
    print(f"⚠️ Still {blank_count} blank sectors — check tickers")

Blank sectors remaining: 52903
⚠️ Still 52903 blank sectors — check tickers


In [0]:
spark.sql("""
    SELECT ticker, COUNT(*) as cnt
    FROM precursor.silver.joined
    WHERE sector IS NULL OR sector = ''
    GROUP BY ticker
    ORDER BY cnt DESC
""").display()

ticker,cnt
HOLX,1568
LW,1558
PAYC,1541
LKQ,1498
MHK,1498
K,1491
IPG,1482
EMN,1465
KMX,1463
MKTX,1434


In [0]:
from pyspark.sql import functions as F

SECTOR_PATCHES = {
    "HOLX": "Health Care", "LW": "Consumer Staples",
    "PAYC": "Information Technology", "LKQ": "Consumer Discretionary",
    "MHK": "Consumer Discretionary", "K": "Consumer Staples",
    "IPG": "Communication Services", "EMN": "Materials",
    "KMX": "Consumer Discretionary", "MKTX": "Financials",
    "WBA": "Consumer Staples", "HES": "Energy",
    "ANSS": "Information Technology", "JNPR": "Information Technology",
    "DFS": "Financials", "BWA": "Consumer Discretionary",
    "FMC": "Materials", "CE": "Materials",
    "TFX": "Health Care", "QRVO": "Information Technology",
    "MRO": "Energy", "AAL": "Industrials",
    "ENPH": "Information Technology", "CZR": "Consumer Discretionary",
    "MTCH": "Communication Services", "RHI": "Industrials",
    "ILMN": "Health Care", "CMA": "Financials",
    "PXD": "Energy", "BIO": "Health Care",
    "XRAY": "Health Care", "VFC": "Consumer Discretionary",
    "CTLT": "Health Care", "ZION": "Financials",
    "WHR": "Consumer Discretionary", "MOH": "Health Care",
    "ETSY": "Consumer Discretionary", "ALK": "Industrials",
    "SEE": "Materials", "ATVI": "Communication Services",
    "DXC": "Information Technology", "NWL": "Consumer Staples",
    "LNC": "Financials", "AAP": "Consumer Discretionary",
    "DISH": "Communication Services", "FRC": "Financials",
    "LUMN": "Communication Services", "SIVB": "Financials",
    "BBWI": "Consumer Discretionary", "VNO": "Real Estate",
    "ABMD": "Health Care", "FBHS": "Industrials",
    "TWTR": "Communication Services", "NLSN": "Industrials",
    "DRE": "Real Estate", "CTXS": "Information Technology",
    "PVH": "Consumer Discretionary", "IPGP": "Information Technology",
    "UA": "Consumer Discretionary", "UAA": "Consumer Discretionary",
    "CERN": "Health Care", "OGN": "Health Care",
    "CDAY": "Information Technology", "DISCK": "Communication Services",
    "DISCA": "Communication Services", "PBCT": "Financials",
    "INFO": "Financials", "XLNX": "Information Technology",
    "GPS": "Consumer Discretionary", "DAY": "Industrials",
    "SEDG": "Information Technology", "LEG": "Consumer Discretionary",
    "WU": "Information Technology", "HBI": "Consumer Discretionary",
    "KSU": "Industrials", "UNM": "Financials",
    "PRGO": "Health Care", "NOV": "Energy",
    "MXIM": "Information Technology", "ALXN": "Health Care",
    "PENN": "Consumer Discretionary", "HFC": "Energy",
    "FLIR": "Information Technology", "VAR": "Health Care",
    "SBNY": "Financials", "XRX": "Information Technology",
    "SLG": "Real Estate", "FLS": "Industrials",
    "FTI": "Energy", "CXO": "Energy",
    "TIF": "Consumer Discretionary", "AIV": "Real Estate",
    "NBL": "Energy", "ETFC": "Financials",
    "KSS": "Consumer Discretionary", "HRB": "Consumer Discretionary",
    "COTY": "Consumer Staples", "JWN": "Consumer Discretionary",
    "HOG": "Consumer Discretionary", "ADS": "Information Technology",
    "VNT": "Information Technology", "HP": "Energy",
    "CPRI": "Consumer Discretionary", "AGN": "Health Care",
    "M": "Consumer Discretionary", "RTN": "Industrials",
    "ARNC": "Materials", "AMTM": "Industrials",
    "XEC": "Energy", "SOLS": "Health Care",
    "WCG": "Health Care", "MBC": "Materials"
}

# Read silver table
df = spark.read.table("precursor.silver.joined")
print(f"Rows before patch: {df.count()}")

# Apply all patches
patch_expr = F.col("sector")
for ticker, sector in SECTOR_PATCHES.items():
    patch_expr = F.when(
        (F.col("ticker") == ticker) &
        (F.col("sector").isNull() | (F.col("sector") == "")),
        sector
    ).otherwise(patch_expr)

df = df.withColumn("sector", patch_expr)

# Verify
blank_count = df.filter(
    F.col("sector").isNull() | (F.col("sector") == "")
).count()
print(f"Blank sectors remaining: {blank_count}")

# Overwrite if clean
if blank_count == 0:
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("mergeSchema", "true") \
      .saveAsTable("precursor.silver.joined")
    print("✅ All sectors patched — silver table updated")
else:
    # Show remaining blanks
    df.filter(
        F.col("sector").isNull() | (F.col("sector") == "")
    ).select("ticker").distinct().display()

Rows before patch: 793717
Blank sectors remaining: 0
✅ All sectors patched — silver table updated


In [0]:
spark.sql("""
    SELECT sector, COUNT(*) as cnt,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct
    FROM precursor.silver.joined
    GROUP BY sector
    ORDER BY cnt DESC
""").display()

sector,cnt,pct
Industrials,118863,15.0
Financials,116890,14.7
Information Technology,104730,13.2
Health Care,99263,12.5
Consumer Discretionary,85097,10.7
Consumer Staples,58767,7.4
Real Estate,48624,6.1
Utilities,46859,5.9
Materials,44132,5.6
Energy,36329,4.6
